# **DLO-JZ Optimisation de l'apprentissage - Jour 2**
<img src="./images/optimisation.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-success">
    
## Objet des notebooks

Le but de ces trois *notebooks* est d'optimiser un code d'apprentissage d'un modèle *Resnet-50* sur *Imagenet* pour Jean Zay en implémentant :
* **TP2.1** : l'optimisation du *Dataloader*
* **TP2.2** : la DDP (*Distributed Data Parallelism*)
* **TP2.3** : la DDP et le problème des paramètres non utilisés


Les cellules dans ce *notebook* ne sont pas prévues pour être modifiées, sauf rares exceptions indiquées dans les commentaires. Les TP se feront en modifiant les codes `dlojz1_X.py`.

Les directives de modification seront marquées par l'étiquette suivante <div class="alert alert-block alert-warning">**TODO**</div>
s solutions sont présentes dans le répertoire `solutions/`.

*Notebook rédigé par l'équipe assistance IA de l'IDRIS, mai 2026.*

</div>

### **Environnement de calcul**

Les fonctions *python* de gestion de queue SLURM développées par l'IDRIS et les fonctions dédiées à la formation DLO-JZ sont à importer.

Le module d'environnement pour les *jobs* et la taille des images sont fixés pour ce *notebook*.
<div class="alert alert-block alert-warning">
    
**TODO :** choisir un pseudonyme (maximum 5 caractères) pour vous différencier dans la queue SLURM pendant la formation.

</div>

In [ ]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import controle_technique, compare, comm_profiler, turbo_profiler, BatchNorm_view, metric_compute_log
MODULE = 'pytorch-gpu/py3/2.8.0'
image_size = 224
account = 'for@a100'
name = 'pseudo'   ## Pseudonyme à choisir
!mkdir -p checkpoints # Création d'un répertoire `checkpoints/` si cela n'a pas déjà été fait.

### **Gestion de la queue SLURM**

Pour afficher vos jobs dans la queue SLURM :

In [ ]:
display_slurm_queue(name)

**Remarque**: cette fonction sera utilisée plusieurs fois dans ce *notebook*. Elle permet d'afficher la queue de manière dynamique, rafraichie toutes les 5 secondes. Elle ne s'arrête que lorsque la queue est vide. Si vous désirez reprendre la main sur le *notebook*, il vous suffira d'arrêter manuellement la cellule avec le bouton *stop*. Cela n'a bien sûr aucun impact sur les *jobs* soumis.

Si vous voulez retirer TOUS vos *jobs* de la queue SLURM, décommenter et exécuter la cellule suivante :

In [ ]:
#!scancel -u $USER

Si vous voulez retirer UN de vos *jobs* de la queue SLURM, décommenter, compléter et exécuter la cellule suivante :

In [ ]:
#!scancel <jobid>

<div class="alert alert-block alert-success">
    
# TP2.1: Optimisation du DataLoader

Dans ce TP, on utilisera le script [dlojz1_1.py](./dlojz1_1.py) dans lequel le profiler PyTorch n'est pas implémenté. Ce script est identique à la solution du TP2_1.

Dans un premier temps, on va désactiver toutes les optimisations du DataLoader (**version sous-optimisée**). Ensuite,  nous pourrons observer l'impact de chacune des optimisations possibles en les réintégrant une par une.

## Découverte de turbo_profiler
Pour ce TP, nous avons implémenté un profiler maison léger `turbo_profiler` basé sur l'outil `Chronometer` pour visualiser le temps passé sur CPU (DataLoader) et sur GPU (le reste de l'itération). Ce profiler est moins précis mais cela nous permettra de désactiver le profiler PyTorch pour ne pas dégrader les performances et éviter de devoir ouvrir l'outil graphique TensorBoard à chaque fois pour visualiser les informations qui nous intéressent.

</div>

<div class="alert alert-block alert-info">

## Version sous-optimisée
<div class="alert alert-block alert-warning">
    
**TODO** : lancer l'exécution sur 1 GPU et 50 itérations (`--test-nsteps 50`) pour passer un contrôle technique qui servira de référence. Cela va prendre quelques minutes (~5min), **vous pouvez passer à la suite sans attendre la fin de l'exécution**.

</div>
</div>

## Garage - Mise à niveau
On fixe la taille d'image pour ce TP et le batch size optimal d'après les expériences du Jour 1

In [ ]:
image_size = 224
bs_optim = 512

### Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
n_gpu = 1
command = f'./dlojz1_1.py -b {bs_optim} --image-size {image_size} --test --test-nsteps 50'
command += f' --num-workers 0 --no-persistent-workers --no-pin-memory --no-non-blocking'
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name, account=account, time_max='01:00:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

<div class="alert alert-block alert-warning">

### **Quizz**

L'éxécution étant assez longue, un quizz vous attend : [Quizz TP2.1](https://www.deepmama.com/quizz/dlojz_quizz4.html)

</div>

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-warning">
    
**TODO** : visualiser la sortie de `turbo_profiler`

</div>

In [ ]:
# call turbo_profiler
dataloader_trial = turbo_profiler(jobid,dataloader_info=True)

Via le turbo profiler, on va également récupérer et stocker les performances obtenues dans une DataFrame `dataloader_trials`
### Initialisation de la DataFrame :

In [ ]:
import pandas as pd
dataloader_trials = pd.DataFrame({"jobid":pd.Series([],dtype=str),
                                  "num_workers":pd.Series([],dtype=int),
                                  "pin_memory":pd.Series([],dtype=str),
                                  "non_blocking":pd.Series([],dtype=str),
                                  "prefetch_factor":pd.Series([],dtype=int),
                                  "persistent_workers":pd.Series([],dtype=str),
                                  "drop_last":pd.Series([],dtype=str),
                                  "loading_time":pd.Series([],dtype=float),
                                  "1st_step_loading_time":pd.Series([],dtype=float),
                                  "CPU_memory_usage(GB)":pd.Series([],dtype=float)})

### Stockage du résultat précédent et visualisation de la *DataFrame* :

In [ ]:
dataloader_trials = pd.concat([dataloader_trials,dataloader_trial])
dataloader_trials.sort_values("loading_time").drop_duplicates(keep="first", ignore_index=True) # trié par ordre croissant du LOADING_TIME

<div class="alert alert-block alert-info">

## Exploration des paramètres d'optimisation du DataLoader
L'objectif de ce TP est de réduire le temps passé sur CPU par le DataLoader.

Pour cette étude, on continue à lancer les exécutions sur 1 GPU et 50 itérations seulement (`--test-nsteps 50`) pour avancer plus rapidement. 

Les différentes optimisations proposées par le DataLoader de PyTorch sont accessibles dans le script `dlojz.py` via les arguments :
* `--num-workers <num_workers>` (défaut à `8`)
* `--persistent-workers` (défaut) ou `--no-persistent-workers`
* `--pin-memory` (défaut) ou `--no-pin-memory`
* `--non-blocking` (défaut) ou `--no-non-blocking`
* `--prefetch-factor <prefetch_factor>` (défaut à `2`, doit être >0)
* `--drop-last` ou `--no-drop-last` (défaut)

</div>

<div class="alert alert-block alert-warning">

**TODO** : répéter les étapes **1 à 4** suivantes en faisant varier ces différents paramètres et observer leurs effets grâce au profiler `turbo_profiler`. Pour comparer les différents essais, ceux-ci seront stockés dans la *DataFrame* `dataloader_trials` initialisée plus tôt.

</div>

#### 1. Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_1.py -b {bs_optim} --image-size {image_size} --test --test-nsteps 50'

# paramètres d'entrée correspondant aux optimisations du DataLoader
command += ' --num-workers 8' 
command += ' --no-pin-memory'
command += ' --no-non-blocking'
command += ' --prefetch-factor 2'
command += ' --no-persistent-workers'
command += ' --no-drop-last'

n_gpu = 1
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

In [ ]:
display_slurm_queue(name)

#### 2. Visualiser le retour du turbo profiler :

In [ ]:
dataloader_trial = turbo_profiler(jobid, dataloader_info=True)

#### 3. Stocker le nouveau résultat et visualiser l'ensemble des runs :

In [ ]:
dataloader_trials = pd.concat([dataloader_trials,dataloader_trial], ignore_index=True)
dataloader_trials.sort_values("loading_time").drop_duplicates(keep="first", ignore_index=True) # trié par ordre croissant du LOADING_TIME

<div class="alert alert-block alert-info">

## **Version optimisée** - Contrôle technique 

<div class="alert alert-block alert-warning">
    
**TODO** : relancer l'exécution sur 1 GPU et 100 itérations (`--test-nsteps 100`) sans profiling pour passer un nouveau contrôle technique, à comparer avec celui de référence.

</div>
</div>

### Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_1.py -b {bs_optim} --image-size {image_size} --test --test-nsteps 100'

# définir ici les paramètres optimaux
command += ' --num-workers ' 
command += ' --no-pin-memory'
command += ' --no-non-blocking'
command += ' --prefetch-factor 2'
command += ' --no-persistent-workers'
command += ' --no-drop-last'

n_gpu = 1
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
turbo_profiler(jobid)

In [ ]:
controle_technique(jobid)

<div class="alert alert-block alert-info">

## **OPTIONNEL** : Visualisation des traces profiler avec TensorBoard (version sous optimisée)

<div class="alert alert-block alert-warning">

**TODO** : relancer le job en **réactivant le profiler PyTorch** dans le script [dlojz1_1.py](./dlojz1_1.py) (revoir le TP1_4) afin de visualiser les traces sous TensorBoard, et les comparer avec la version optimisée étudiée dans le TP1.4.

</div>
</div>

### Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'./dlojz1_1.py -b {bs_optim} --image-size {image_size} --test --test-nsteps 8'
command += f' --num-workers 0 --no-persistent-workers --no-pin-memory --no-non-blocking'

n_gpu = 1
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

<div class="alert alert-block alert-warning">

**TODO** : vérifier qu'une trace a bien été générée dans le répertoire `profiler/<name>_<jobid>_bs512_is224/` sous la forme d'un fichier `.json`:

</div>

In [ ]:
!tree profiler/

<div class="alert alert-block alert-warning">

**TODO** : visualiser cette trace grâce à l'application TensorBoard. 

</div>

<div class="alert alert-block alert-danger">

**IMPORTANT** : une fois le TP terminé, penser à quitter l'instance JupyterHub pour **libérer le GPU** ( *> Hub Control Panel > Cancel* ).

</div>

<img src="./images/stop.png" style="float: left; margin-right: 1em;"/>